# 🧭 Agoda Data Engineering — System Design Interview Prep
### Round 2: Scalable, Robust & Maintainable Data Engineering Systems

> **Context:** Agoda processes **1.8 Trillion Kafka events/day**, **12 PB/month**,
> runs **1000 simultaneous A/B experiments**, operates across 3.6M+ properties.
> Real stack: Kafka · Spark · StarRocks · Kubeflow · MLflow · HDFS · Impala · Vertica · Elasticsearch.

---

## 📚 Table of Contents

1. [Agoda's Real Architecture & Scale](#1)
2. [Most Likely Interview Questions (OTA-Specific)](#2)
3. [End-to-End System Design: Hotel Search & Pricing Pipeline](#3)
4. [Real-Time vs Batch: Lambda & Kappa Architectures](#4)
5. [Apache Kafka at Scale (1.8T events/day)](#5)
6. [Apache Spark: Deep Dive with Code](#6)
7. [Data Modeling: OTA Domain (SQL + NoSQL)](#7)
8. [Feature Store Design for ML Pipelines](#8)
9. [Data Warehouse & OLAP (StarRocks / Impala)](#9)
10. [Data Quality, Observability & SLAs](#10)
11. [Distributed Storage: Partitioning, Indexing, Caching](#11)
12. [Operational Excellence: Monitoring, DR, Graceful Degradation](#12)
13. [Common Mistakes, Tricks & Edge Cases](#13)
14. [System Design Templates — Whiteboard Approach](#14)
15. [Mock Interview Q&A Bank](#15)


---
<a id='1'></a>
## 1. Agoda's Real Architecture & Scale

Understanding what Agoda *actually* runs is your single biggest advantage — the interviewer works there.

### Key Numbers to Internalize

| Metric | Value |
|--------|-------|
| Kafka events/day | **1.8 Trillion** (2023, 2× YoY growth) |
| Data processed/month | **12 Petabytes** |
| Simultaneous A/B experiments | **1,000** |
| Properties on platform | **3.6 Million+** |
| Countries | **195+** |
| Data/ML engineering headcount | **~300** |
| Kafka p99 latency (analytics path) | **10 seconds** |

### Agoda's Actual Tech Stack

```
Ingestion Layer:   Kafka (1.8T events/day, MULTI-CLUSTER per use-case)
                   2-Step Logging: App → Disk → Forwarder → Kafka
Processing:        Apache Spark (batch + streaming), Livy, Yarn
Storage:           HDFS (data lake), VastStorage
OLAP / Query:      StarRocks, Impala, Vertica, Elasticsearch
ML Platform:       Kubeflow (orchestration), MLflow (experiment tracking)
Monitoring:        Grafana + White Falcon (custom in-house TSDB)
                   GoFresh (freshness/SLA monitor), JMXTrans → Graphite
Data Discovery:    Data Market (internal catalog), Schema Registry (Kafka)
Visualization:     Metabase, Tableau, custom funnel dashboards
Languages:         Kotlin, Scala, Python, Go, TypeScript
```

### The 2-Step Logging Architecture (KEY Agoda Pattern)

```
[Application Server]  ← never blocks on Kafka
      │
      ▼  (write to local disk — fire and forget, < 1ms)
[Local Disk / Ring Buffer]
      │
      ▼  (Forwarder daemon — reads, batches, sends)
[Kafka Cluster — Analytics]
      │
      ├──► [Spark Streaming]  ──► [HDFS / Data Lake]
      ├──► [Spark Batch ETL]  ──► [StarRocks / Vertica]
      └──► [ML Feature Pipeline] ──► [Kubeflow / MLflow]
```

**Why this matters for the interview:**
This pattern trades ~10 s latency for resiliency. Apps never block on Kafka.
The Forwarder handles retries, backpressure, and file rotation independently.
This is the correct answer when asked *"how do you guarantee delivery without impacting application latency?"*

### Multi-Cluster Kafka Strategy (Blast-Radius Isolation)

| Cluster | Use-case | Latency SLA |
|---------|----------|-------------|
| analytics-cluster | clickstream, search events | p99 ≤ 10 s |
| booking-cluster | reservations, payments | p99 ≤ 500 ms |
| ml-cluster | feature pipeline events | p99 ≤ 5 s |
| audit-cluster | message-count auditing only | p99 ≤ 30 s |
| replication-cluster | cross-DC sync | p99 ≤ 2 s |

**Benefit:** A runaway consumer on the analytics cluster cannot starve the booking cluster.


---
<a id='2'></a>
## 2. Most Likely Interview Questions — OTA / Agoda Specific

Compiled from Glassdoor (2024–2026), Blind, and Agoda Engineering blog patterns.

### 🔴 Tier 1 — Very Likely (design these end-to-end)

1. **Design Agoda's Hotel Ranking Data Pipeline**
   Inputs: search events, booking history, price data, reviews, inventory.
   Output: ranked hotel list per query, updated near-real-time.
   Key challenge: freshness vs. cost, consistency at 1 M+ searches/day.

2. **Design a Real-Time Dynamic Pricing System**
   Inputs: demand signals, competitor prices, inventory, historical bookings.
   Output: price per room-type per date, updated every 5 minutes.
   Key challenge: write amplification (3.6 M hotels × 365 dates × N room types).

3. **Design a Unified Financial Data Pipeline (FINUDP pattern)**
   Consolidate 3 independent pipelines into one source of truth.
   SLA: hourly updates, auto-halt on quality failure.
   Challenge: reduce runtime from 5 hours → 30 minutes.

4. **Design a Feature Store for ML (pricing, ranking, personalization)**
   Offline store (training) + Online store (sub-10 ms serving).
   Point-in-time correctness, backfill, feature versioning.

5. **Design a Hotel Search System at 1 M+ QPS**
   Availability, pricing, ranking, personalization — all in < 200 ms p99.

### 🟡 Tier 2 — Likely

6. A/B Testing Data Platform (Agoda runs 1000 simultaneous experiments)
7. Customer Clickstream Analytics Pipeline (session reconstruction)
8. Data Quality Monitoring System (ML-based anomaly detection on pipelines)
9. Booking Event Processing System (exactly-once semantics)
10. Hotel Inventory Management System (high write, strong consistency)

### ⚡ Clarifying Questions — Always Ask These First

```
1. What is the expected QPS / data volume?
2. What latency is acceptable — real-time (<1s), near-real-time (<1min), batch (hourly/daily)?
3. What consistency level — eventual, strong, or read-your-writes?
4. Who are the consumers — ML models, dashboards, APIs, or downstream services?
5. What is the SLA for data freshness? What happens if we miss it?
6. Is this write-heavy or read-heavy? What is the read/write ratio?
7. Do we need to support reprocessing / backfill?
8. What is the data retention requirement?
```


---
<a id='3'></a>
## 3. End-to-End System Design: Hotel Search & Pricing Pipeline

### Problem Statement
Design a data pipeline for Agoda's hotel search:
- 1 M+ search queries / day
- Returns ranked results with live pricing in < 200 ms p99
- Prices and availability updated every 5 minutes
- 3.6 M hotels × 365 dates × 10 room types = **13 B price points**

### Back-of-Envelope Estimation (do this live!)

```
Price points:    3.6M × 365 × 10        = 13.1 Billion
Storage (raw):   13.1B × 200 bytes       = 2.62 TB → too big for Redis
Hot window:      13.1B × (90/365) × 0.30 = ~970 M keys × 200 B = ~194 GB → feasible!
                 (90-day horizon × ~30% availability rate)

Search QPS:      1M queries/day = ~12 QPS avg, ~120 QPS peak (10× factor)
Price updates:   3.6M hotels / 5min = 12,000 hotel price-sets/sec during batch
Kafka events:    12,000 hotels/sec × 10 room types × 365 dates = 43.8M msgs/sec during update
                 → Requires 4-5 Kafka clusters to handle this burst
```

### High-Level Architecture

```
  SOURCES                     INGESTION            PROCESSING           SERVING

  Supplier APIs  ──────────►  Kafka                Spark Streaming  ──► Redis Cluster
  (price feeds)               hotel.prices         (5-min windows)      (hot prices)

  User Actions  ──────────►  Kafka                 Spark Batch      ──► StarRocks
  (clicks, search)            user.search           (hourly ETL)         (analytics)

  Booking Events ─────────►  Kafka                 ML Ranking       ──► Elasticsearch
                              hotel.bookings        (offline daily)      (search index)

  Review Updates ─────────►  Kafka                                  ──► Kubeflow
                              hotel.reviews                              (ML training)
```

### Key Design Decisions & Trade-offs

| Decision | Option A | Option B | Chosen & Why |
|----------|----------|----------|--------------|
| Price storage | Redis (all 13B points) | Redis (hot) + Vertica (cold) | **Hybrid** — 90-day hot window in Redis (~194 GB feasible) |
| Price update frequency | Real-time push | 5-min micro-batch | **5-min batch** — supplier APIs don't refresh faster; reduces write amplification |
| Ranking | Pre-computed offline | Real-time scoring | **Hybrid** — base scores offline daily, personalization features real-time |
| Consistency | Strong (all replicas) | Eventual (faster) | **Eventual** — 5-min stale prices are acceptable for search results |
| Search index | Elasticsearch | Custom inverted index | **Elasticsearch** — handles geo-search, faceting, text natively |

### Price Storage Key Design

```
Redis Hash pattern (efficient for room-type fan-out):

  HSET hotel:12345:prices:2025-06-01  STANDARD 4500  DELUXE 6200  SUITE 12000
  EXPIRE hotel:12345:prices:2025-06-01  900     ← 15-min TTL; refreshed by Spark job

Read path (hotel search API, < 5 ms):
  HGETALL hotel:12345:prices:2025-06-01
  → {STANDARD: 4500, DELUXE: 6200, SUITE: 12000}

Bulk read (search results page, 20 hotels):
  pipeline = redis.pipeline()
  for hotel_id in top_20_hotel_ids:
      pipeline.hgetall(f'hotel:{hotel_id}:prices:{check_in}')
  prices = pipeline.execute()    ← single round trip
```

### Failure Modes & Mitigations

| Failure | Impact | Mitigation |
|---------|--------|------------|
| Redis down | No live prices | Serve prices from StarRocks (stale but available) + show "price as of" |
| Kafka lag > 30 min | Prices 30+ min stale | Alert + extend Redis TTL + serve stale with warning |
| Supplier API down | No price updates | Serve last-known price; flag hotels as "price may vary" |
| ML ranking down | Unranked results | Fall back to rule-based: sort by (review_score DESC, price ASC) |
| Elasticsearch down | No search results | Fall back to StarRocks full-text search (slower but available) |


---
<a id='4'></a>
## 4. Real-Time vs Batch: Lambda & Kappa Architectures

### Lambda Architecture

```
                   ┌────────────────────────────────┐
Raw Data ─────────►│         SOURCE (Kafka)          │
                   └────────────┬──────────┬─────────┘
                                │          │
                   ┌────────────▼──┐  ┌────▼────────────┐
                   │  BATCH LAYER  │  │  SPEED LAYER     │
                   │  (Spark)      │  │  (Spark Stream)  │
                   │               │  │                  │
                   │  Full recomp  │  │  Incremental     │
                   │  High latency │  │  Low latency     │
                   │  Accurate     │  │  Approximate     │
                   └────────────┬──┘  └────┬────────────┘
                                │          │
                   ┌────────────▼──────────▼────────────┐
                   │          SERVING LAYER              │
                   │  Merge batch view + speed view      │
                   └─────────────────────────────────────┘
```

**Use when:** You need both historical-reprocessing accuracy AND low-latency fresh data.
**OTA example:** Hotel demand forecasting (batch, full history) merged with live booking rate (stream).

**Pros:** Accurate batch results; speed layer handles freshness gap.
**Cons:** Dual codebase — batch and stream logic MUST produce identical results (hard to guarantee).

### Kappa Architecture

```
Raw Data ──► Kafka (long retention) ──► Unified Stream Processing ──► Serving Layer
                        │                          │
                        └──► Replay to backfill    └──► Single codebase
                             (no batch layer needed)     (Flink or Spark Structured Streaming)
```

**Use when:** Your stream processing engine is powerful enough to handle historical computation too.
**OTA example:** Agoda's FINUDP — one unified Spark pipeline replaced three separate pipelines.

**Pros:** Single codebase, simpler ops, backfill = replay Kafka topic.
**Cons:** Requires Kafka to retain data long enough — expensive at 1.8 T events/day.

### ⚠️ Common Mistake in Interviews

> *'I will use Lambda because it's the industry standard.'*

**Better answer:** *'Lambda introduces dual-maintenance complexity — the batch and stream
codepaths must always agree. For this pipeline I'd prefer Kappa with Spark Structured
Streaming, which handles both real-time and reprocessing from the same code.
I'd only fall back to Lambda if the stream engine demonstrably can't handle the
full historical computation within our SLA.'*

### OTA Design Choice

```python
# Agoda scale: Kappa preferred for analytics pipelines
# Lambda useful for: ML training (needs full history) + real-time scoring (needs low latency)

# Hotel Ranking hybrid example:
# Batch (daily, offline):  base_score  = f(reviews, historical_bookings, distance, amenities)
# Stream (5-min, online):  demand_boost = f(current_searches, recent_bookings, inventory)
# Merge at serving time:   final_score = base_score * 0.7 + demand_boost * 0.3
```


---
<a id='5'></a>
## 5. Apache Kafka at Scale (1.8 T events/day = ~20 M events/second)

### Core Kafka Concepts — Know Cold

| Concept | What it means | OTA relevance |
|---------|---------------|---------------|
| **Partition** | Unit of parallelism; ordered log | `hotel_id % N` for per-hotel ordering |
| **Consumer Group** | Multiple consumers sharing a topic | ML pipeline + BI pipeline consume same topic independently |
| **Offset** | Position in partition log | Checkpoint for exactly-once processing |
| **Replication Factor** | N copies across brokers | RF=3 for bookings, RF=2 for analytics |
| **ISR** | In-sync replicas | If ISR < min.insync.replicas, producer blocks/fails |
| **Retention** | How long data is kept | Analytics: 7 days. Audit: 30 days. Replay: 90 days |
| **Log Compaction** | Keep only latest value per key | Price topic: always want latest price per hotel+date key |
| **Schema Registry** | Enforce Avro/Protobuf schema | Prevents bad producers from breaking downstream consumers |

### Delivery Semantics — The Interview Trap

```
AT MOST ONCE   → acks=0, no retries. Fire-and-forget.
               Use: high-volume click events (losing 0.1% is acceptable)

AT LEAST ONCE  → acks=all + retries. Consumer may see duplicates on crash.
               Use: analytics events (make consumer idempotent by de-duping on event_id)

EXACTLY ONCE   → Kafka Transactions. End-to-end exactly-once effect.
               Use: booking/payment events — duplicate booking = disaster
               Config: enable.idempotence=true + transactional.id
```

### Multi-Cluster vs Single Cluster — Agoda's Choice

```
NAIVE:  One giant Kafka cluster for everything
        Problem: analytics runaway consumer starves booking consumer
                 DLQ flooding one cluster blocks all pipelines

AGODA:  Separate clusters per use-case
        analytics-cluster  → search/click events (high volume, lossy OK)
        booking-cluster    → reservations/payments (low volume, exactly-once)
        ml-cluster         → feature pipeline events (medium volume, freshness SLA)
        audit-cluster      → message-count audits only (dedicated)
        replication-cluster → cross-DC Kafka MirrorMaker 2

Benefit: Blast radius isolation. Testing a new consumer group can't tank production.
```

### Kafka Auditing Pattern (Agoda's Actual Implementation)

```
Client Library (producer side)
  └── background thread aggregates message counts per time bucket
  └── sends audit events to dedicated audit-cluster

Audit Cluster stores counts:
  {topic, partition, time_bucket, produced_count, consumed_count}

GoFresh / White Falcon monitors:
  IF produced_count != consumed_count within time_bucket window → ALERT
  This gives end-to-end data completeness guarantees.
```

### Schema Evolution Rules

```
SAFE (backward compatible):
  + Add new field WITH a default value
  + Add new optional (nullable) field
  + Rename a type alias (not field name)

BREAKING (never do in production without migration plan):
  - Remove a required field
  - Change field type (int → string)
  - Rename a field
  - Add required field WITHOUT default

AGODA's rule: 30-day advance notice to all consumers before any breaking change.
Use Schema Registry compatibility mode = BACKWARD
```


In [ ]:
# ============================================================
# Kafka: Partitioning Strategy Demo
# ============================================================
import hashlib

def kafka_partition(key: str, num_partitions: int) -> int:
    '''Simulates Kafka's default murmur2 hash partitioning.'''
    h = int(hashlib.md5(key.encode()).hexdigest(), 16)
    return h % num_partitions

NUM_PARTITIONS = 64  # typical for a high-volume topic

# Scenario 1: Random partitioning (no key) — bad for ordering
import random
print('=== No key (round-robin) ===')
for _ in range(5):
    print(f'  Random partition: {random.randint(0, NUM_PARTITIONS-1)}')

# Scenario 2: Partition by hotel_id — ordering within hotel guaranteed
print('\n=== Key = hotel_id (consistent mapping) ===')
for hotel_id in [12345, 99999, 50001, 777, 12345]:  # 12345 appears twice
    p = kafka_partition(str(hotel_id), NUM_PARTITIONS)
    print(f'  hotel {hotel_id:>6} → partition {p:>3}')
# Note: hotel 12345 always maps to the same partition!

# Scenario 3: Hot partition problem
# Bangkok's top hotel gets 1000x more price updates than average
# Solution: compound key to distribute load
print('\n=== Compound key (hotel_id + date_bucket) — avoids hot partition ===')
hot_hotel = 1  # hypothetical #1 hotel
for date_bucket in ['2025-06-01', '2025-06-02', '2025-06-03', '2025-06-04']:
    key = f'{hot_hotel}:{date_bucket}'
    p = kafka_partition(key, NUM_PARTITIONS)
    print(f'  key={key!r} → partition {p:>3}')

print('\n=== Consumer Lag Monitoring (concept) ===')
partitions = {
    0: {'latest': 1_500_000, 'committed': 1_499_800},
    1: {'latest': 1_800_000, 'committed': 1_400_000},  # lagging!
    2: {'latest': 1_200_000, 'committed': 1_199_950},
}
LAG_THRESHOLD = 50_000
for p, offsets in partitions.items():
    lag = offsets['latest'] - offsets['committed']
    status = 'ALERT 🔴' if lag > LAG_THRESHOLD else 'OK ✅'
    print(f'  Partition {p}: lag={lag:>10,}  {status}')


In [ ]:
# ============================================================
# Agoda's 2-Step Logging Architecture — Python simulation
# ============================================================
import json, time, threading, queue, os, tempfile

class AgodaEventLogger:
    '''Step 1: Application writes to local disk — never blocks on Kafka.'''

    def __init__(self, log_dir=None):
        self.log_dir = log_dir or tempfile.mkdtemp()
        self._lock = threading.Lock()
        self._file = None
        self._bytes_written = 0
        self._rotate_threshold = 512 * 1024  # 512 KB
        self._open_new_file()

    def log(self, event: dict):
        '''Fire-and-forget. Latency: < 1 ms (local disk write).'''
        line = json.dumps(event) + '\n'
        with self._lock:
            self._file.write(line)
            self._bytes_written += len(line)
            if self._bytes_written >= self._rotate_threshold:
                self._rotate()

    def _open_new_file(self):
        fname = os.path.join(self.log_dir, f'events_{int(time.time()*1000)}.jsonl')
        self._file = open(fname, 'w')
        self._bytes_written = 0

    def _rotate(self):
        self._file.flush()
        self._file.close()
        self._open_new_file()


class KafkaForwarder:
    '''Step 2: Separate daemon reads local files and forwards to Kafka.'''

    def __init__(self, log_dir, kafka_topic):
        self.log_dir = log_dir
        self.kafka_topic = kafka_topic
        self.sent_count = 0
        self.failed_count = 0

    def forward_all(self):
        for fname in sorted(os.listdir(self.log_dir)):
            if not fname.endswith('.jsonl'):
                continue
            fpath = os.path.join(self.log_dir, fname)
            self._send_file(fpath)

    def _send_file(self, fpath):
        with open(fpath) as f:
            for line in f:
                event = json.loads(line)
                self._send_to_kafka(event)

    def _send_to_kafka(self, event):
        # In production: KafkaProducer.send() with retries
        # Trade-off: p99 latency ~10s for analytics (disk → forwarder → kafka)
        # For booking events: bypass to Kafka directly for < 500ms latency
        self.sent_count += 1


# Demo
logger = AgodaEventLogger()
events_to_log = [
    {'type': 'hotel_search', 'user_id': 42, 'destination': 'Bangkok', 'ts': time.time()},
    {'type': 'hotel_view',   'user_id': 42, 'hotel_id': 12345,        'ts': time.time()},
    {'type': 'price_check',  'user_id': 42, 'hotel_id': 12345,        'ts': time.time()},
    {'type': 'booking_init', 'user_id': 42, 'hotel_id': 12345,        'ts': time.time()},
]

for e in events_to_log:
    logger.log(e)

logger._file.flush()

forwarder = KafkaForwarder(logger.log_dir, 'user.search.events')
forwarder.forward_all()

print(f'Events logged to disk: {len(events_to_log)}')
print(f'Events forwarded to Kafka: {forwarder.sent_count}')
print(f'Key insight: Application thread was NEVER blocked by Kafka.')
print(f'For booking events, bypass this and write directly — latency SLA is tighter.')


---
<a id='6'></a>
## 6. Apache Spark: Deep Dive with Code

### Architecture in 30 seconds

```
Driver (SparkContext)
  ├── DAG Scheduler → stages → tasks
  ├── Task Scheduler → assign tasks to executors
  └── Cluster Manager (YARN / Kubernetes)
          ├── Executor 1 (JVM): runs tasks, caches data
          ├── Executor 2 (JVM)
          └── Executor N (JVM)

KEY: Transformations are LAZY (build a DAG). Actions TRIGGER execution.
Narrow transformation: no shuffle (map, filter, union)
Wide transformation:   shuffle needed (groupBy, join, repartition) → stage boundary
```

### Performance Tuning Reference Card

| Symptom | Root Cause | Fix |
|---------|------------|-----|
| One task 100× slower | Data skew | Salting, AQE skew join, broadcast join |
| OOM on executors | Too much data per partition | Increase partitions, reduce `maxOffsetsPerTrigger` |
| Shuffle spill to disk | Not enough memory | Increase `spark.executor.memory`, tune `spark.memory.fraction` |
| Too many small HDFS files | Over-partitioned output | `df.coalesce(N)` before write |
| Slow despite parallelism | Wrong partition count | Rule: 2–4 partitions per CPU core |
| Kafka timeout in streaming | Consumer can't keep up | Reduce `maxOffsetsPerTrigger`, add executors |


In [ ]:
# ============================================================
# SPARK PATTERN 1: Hotel Price Aggregation — Batch ETL
# ============================================================
# Note: Run this with a real SparkSession in your environment.
# The code is self-contained and production-ready.

SPARK_BATCH_CODE = '''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \\
    .appName('HotelPriceAggregation') \\
    .config('spark.sql.shuffle.partitions', '400') \\
    .config('spark.sql.adaptive.enabled', 'true') \\
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \\
    .getOrCreate()

# Read raw price events (partitioned by ingestion date — use partition pruning)
prices_raw = spark.read.parquet('hdfs:///data/prices/ingestion_date=2025-04-17/')

# STEP 1: Deduplication — at-least-once Kafka delivery means duplicates exist
deduped = prices_raw.withColumn(
    'row_num',
    F.row_number().over(
        Window.partitionBy('event_id')
              .orderBy(F.desc('event_timestamp'))
    )
).filter('row_num = 1').drop('row_num')

# STEP 2: Aggregate — min price per (hotel, check_in_date, room_type)
agg_prices = deduped.groupBy('hotel_id', 'check_in_date', 'room_type').agg(
    F.min('price').alias('min_price'),
    F.max('price').alias('max_price'),
    F.avg('price').alias('avg_price'),
    F.count('*').alias('supplier_count'),
    F.max('event_timestamp').alias('last_updated')
)

# STEP 3: Write — partitioned by check_in_date for downstream query pruning
agg_prices.write \\
    .partitionBy('check_in_date') \\
    .mode('overwrite') \\
    .parquet('hdfs:///data/processed/hotel_prices/')
'''

print('=== Batch ETL Pipeline Structure ===')
print(SPARK_BATCH_CODE)


In [ ]:
# ============================================================
# SPARK PATTERN 2: Handling Skewed Data (CRITICAL at OTA scale)
# ============================================================
# Bangkok's top hotel may get 100x more price updates than average.
# This creates a 'hot partition' where one executor does all the work.

SPARK_SKEW_CODE = '''
from pyspark.sql import functions as F

# ---- SOLUTION 1: Manual Salting ----
SALT_FACTOR = 20

# Add random salt to distribute hot keys across partitions
df_salted = df.withColumn(
    'salted_hotel_id',
    F.concat(
        F.col('hotel_id').cast('string'),
        F.lit('_'),
        (F.rand() * SALT_FACTOR).cast('int')
    )
)

# First aggregation: group by salted key (distributes hot partition)
partial = df_salted.groupBy('salted_hotel_id', 'hotel_id') \\
    .agg(F.sum('revenue').alias('partial_revenue'))

# Second aggregation: remove salt
final = partial.groupBy('hotel_id') \\
    .agg(F.sum('partial_revenue').alias('total_revenue'))

# ---- SOLUTION 2: Spark AQE (Spark 3.0+, automatic) ----
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '5')
# AQE auto-detects and splits skewed partitions during execution

# ---- SOLUTION 3: Broadcast join for small dimension tables ----
# Hotel metadata: 3.6M hotels x 200 bytes = ~720 MB — broadcast it!
hotel_meta = spark.read.parquet('hdfs:///data/hotel_metadata/')
events = spark.read.parquet('hdfs:///data/events/')

# Broadcast forces hotel_meta to every executor — no shuffle needed
result = events.join(F.broadcast(hotel_meta), 'hotel_id')
# Reduces a 10-minute shuffle join to a 30-second map-side join
'''

print('=== Data Skew Solutions ===')
print(SPARK_SKEW_CODE)

print('=== Common Spark Mistakes ===')
mistakes = [
    ('Default shuffle partitions = 200', 'At TB scale: use num_cores * 4, e.g. 400-2000'),
    ('Calling .count() mid-pipeline', 'Triggers full execution. Use accumulators instead'),
    ('Python UDF row-by-row', 'Use pandas_udf (vectorized) — 10-100x faster'),
    ('Not caching reused DataFrames', 'df.cache() before second action or full rescan occurs'),
    ('Writing without partitioning', 'Always .partitionBy(date_col) for downstream pruning'),
    ('Fixed maxOffsetsPerTrigger', 'Tune this; too high = OOM, too low = persistent lag'),
]
for mistake, fix in mistakes:
    print(f'  MISTAKE: {mistake}')
    print(f'  FIX:     {fix}')
    print()


In [ ]:
# ============================================================
# SPARK PATTERN 3: Structured Streaming — Near Real-Time Pricing
# ============================================================

STREAMING_CODE = '''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.appName('RealtimePricing').getOrCreate()

# ALWAYS define schema for streaming — never infer (it triggers a job)
price_schema = StructType([
    StructField('hotel_id',       StringType()),
    StructField('room_type',      StringType()),
    StructField('price',          DoubleType()),
    StructField('currency',       StringType()),
    StructField('check_in_date',  StringType()),
    StructField('event_timestamp', LongType()),
])

stream_df = spark.readStream \\
    .format('kafka') \\
    .option('kafka.bootstrap.servers', 'kafka-analytics:9092') \\
    .option('subscribe', 'hotel.prices') \\
    .option('startingOffsets', 'latest') \\
    .option('maxOffsetsPerTrigger', 1_000_000) \\
    .load()

parsed = stream_df \\
    .select(F.from_json(F.col('value').cast('string'), price_schema).alias('d')) \\
    .select('d.*') \\
    .withColumn('event_time', F.from_unixtime('event_timestamp').cast('timestamp'))

# Windowed aggregation with watermark (handles late-arriving data)
windowed = parsed \\
    .withWatermark('event_time', '10 minutes')  # tolerate 10-min late data \\
    .groupBy(
        'hotel_id', 'room_type', 'check_in_date',
        F.window('event_time', '5 minutes')  # 5-min tumbling window
    ) \\
    .agg(F.min('price').alias('min_price_5min'))

# Write to Redis via foreach (or to Kafka for downstream consumers)
def write_to_redis(row):
    import redis
    r = redis.Redis(host='redis-cluster', port=6379)
    key = f"hotel:{row.hotel_id}:prices:{row.check_in_date}:{row.room_type}"
    r.setex(key, 300, str(row.min_price_5min))  # TTL = 5 min

query = windowed.writeStream \\
    .foreach(write_to_redis) \\
    .outputMode('update') \\
    .trigger(processingTime='30 seconds') \\
    .option('checkpointLocation', 'hdfs:///checkpoints/pricing/') \\
    .start()
# Checkpoint = fault tolerance. If job crashes, resumes from last committed offset.
'''

print('=== Spark Structured Streaming Pipeline ===')
print(STREAMING_CODE)


---
<a id='7'></a>
## 7. Data Modeling: OTA Domain (SQL + NoSQL)

### Core OTA Entity Relationships

```
HOTEL ──< ROOM_TYPE ──< PRICE_RATE (per date, per supplier)
  │                   ──< INVENTORY_UNIT
  │
  └──< BOOKING ──> CUSTOMER
  └──< REVIEW  ──> CUSTOMER
  └──< AMENITY (M:N via HOTEL_AMENITY)
  └──< PROPERTY_CONTENT (photos, descriptions)
```

### Star Schema for Analytics (Data Warehouse)

```
              DIM_HOTEL          DIM_CUSTOMER
                  \                  /
                   \                /
                    FACT_BOOKINGS
                   /    |    |    \
                  /     |    |     \
         DIM_DATE   DIM_GEO  DIM_SUPPLIER  DIM_ROOM_TYPE

FACT measures: room_nights, gross_booking_value, net_revenue, discount, is_cancelled

WHY star over snowflake?
  Fewer JOINs → faster StarRocks/Vertica queries
  BI tools (Tableau, Metabase) work best with star schema
  Denormalization is fine for read-only analytical workloads
```

### NoSQL Pattern Selection Guide

| Use Case | Store | Key Design | Why |
|----------|-------|-----------|-----|
| Price cache (hot reads) | Redis | `hotel:{id}:prices:{date}` → Hash | Sub-5ms, supports bulk pipeline |
| Availability (write-heavy) | Cassandra | PK=(hotel_id, room_type_id) CK=check_in_date | Wide-row time-series, fast range scans |
| Hotel search & geo | Elasticsearch | One doc per hotel | Inverted index, geo queries, faceting |
| Booking event log | Kafka + HDFS | Append-only | Immutable audit trail, replay support |
| Session/idempotency | Redis | `session:{id}` with TTL | Auto-expiry, atomic operations |


In [ ]:
# ============================================================
# OTA SQL Schema — OLTP (Booking System)
# ============================================================

schema_ddl = '''
-- Partitioned by hash for even distribution across shards
CREATE TABLE hotels (
    hotel_id        BIGINT PRIMARY KEY,
    name            VARCHAR(255) NOT NULL,
    star_rating     TINYINT,
    country_code    CHAR(2),
    city_id         INT,
    latitude        DECIMAL(9,6),
    longitude       DECIMAL(9,6),
    is_active       BOOLEAN DEFAULT TRUE,
    created_at      TIMESTAMP DEFAULT NOW()
) PARTITION BY HASH(hotel_id) PARTITIONS 256;

-- Price rates — most queried table in the system
CREATE TABLE price_rates (
    rate_id         BIGINT AUTO_INCREMENT,
    hotel_id        BIGINT NOT NULL,
    room_type_id    BIGINT NOT NULL,
    supplier_id     INT NOT NULL,
    check_in_date   DATE NOT NULL,
    check_out_date  DATE NOT NULL,
    price           DECIMAL(12,2) NOT NULL,
    currency        CHAR(3) NOT NULL,
    availability    SMALLINT DEFAULT 0,
    cancellation    ENUM("FREE","PARTIAL","NON_REFUNDABLE"),
    last_updated    TIMESTAMP,

    -- Composite index for the primary search pattern
    INDEX idx_search (hotel_id, check_in_date, check_out_date, room_type_id),

    -- Covering index: query served entirely from index, no heap read
    INDEX idx_price_cover (check_in_date, check_out_date, hotel_id, price, availability)

) PARTITION BY RANGE (TO_DAYS(check_in_date)) (
    PARTITION p_2025_q1 VALUES LESS THAN (TO_DAYS("2025-04-01")),
    PARTITION p_2025_q2 VALUES LESS THAN (TO_DAYS("2025-07-01")),
    PARTITION p_2025_q3 VALUES LESS THAN (TO_DAYS("2025-10-01")),
    PARTITION p_2025_q4 VALUES LESS THAN (TO_DAYS("2026-01-01")),
    PARTITION p_future  VALUES LESS THAN MAXVALUE
);

-- Bookings — immutable event log, never UPDATE or DELETE
CREATE TABLE bookings (
    booking_id      VARCHAR(36) PRIMARY KEY,   -- UUID
    hotel_id        BIGINT NOT NULL,
    room_type_id    BIGINT NOT NULL,
    customer_id     BIGINT NOT NULL,
    check_in_date   DATE NOT NULL,
    check_out_date  DATE NOT NULL,
    total_price     DECIMAL(12,2),
    currency        CHAR(3),
    status          ENUM("CONFIRMED","CANCELLED","CHECKED_IN","NO_SHOW"),
    created_at      TIMESTAMP DEFAULT NOW(),

    INDEX idx_hotel_dates   (hotel_id, check_in_date, status),
    INDEX idx_customer_date (customer_id, created_at)
);
'''

print('=== Key Index Design Decisions ===')
decisions = [
    ('price_rates: range partition by check_in_date',
     'Search queries filter by date range → scan only relevant partitions'),
    ('hotels: hash partition by hotel_id',
     'Even distribution of 3.6M hotels across shards'),
    ('bookings: UUID primary key',
     'Globally unique without centralized sequence — safe for distributed inserts'),
    ('price_rates: covering index on (date, hotel_id, price, availability)',
     'Most search queries served from index alone — no heap I/O'),
    ('Partition key in WHERE: never apply functions to partition column',
     'WHERE check_in_date = x → partition prune. WHERE MONTH(check_in_date) = 6 → FULL SCAN'),
]
for decision, reason in decisions:
    print(f'  DECISION: {decision}')
    print(f'  REASON:   {reason}')
    print()

print('=== Cassandra — Hotel Availability (write-heavy, time-series) ===')
cassandra_ddl = '''
CREATE TABLE hotel_availability (
    hotel_id        INT,
    room_type_id    INT,
    check_in_date   DATE,
    available_rooms INT,
    last_updated    TIMESTAMP,
    PRIMARY KEY ((hotel_id, room_type_id), check_in_date)
) WITH CLUSTERING ORDER BY (check_in_date ASC)
  AND compaction = {
      'class': 'TimeWindowCompactionStrategy',
      'compaction_window_size': 1,
      'compaction_window_unit': 'DAYS'
  };
-- Anti-pattern: querying by check_in_date alone (no partition key = full cluster scan)
-- Always include (hotel_id, room_type_id) in the WHERE clause
'''
print(cassandra_ddl)


---
<a id='8'></a>
## 8. Feature Store Design for ML Pipelines

Agoda runs ML models for: hotel ranking, dynamic pricing, personalization, fraud, demand forecasting.

### Feature Store Architecture

```
WRITE PATH
  Raw Events → Kafka → Spark Feature Pipeline → Offline Store (HDFS Parquet)
                                              → Online Store  (Redis)

READ PATH — Training
  Offline Store → Point-in-Time Join with labels → Training Dataset → Kubeflow/MLflow

READ PATH — Inference (< 10 ms)
  Online Store (Redis) → Feature Server API → ML Model → Prediction

CONSISTENCY GUARANTEE
  Same Spark code writes to BOTH stores → eliminates training/serving skew
```

### Feature Categories for OTA

| Category | Freshness | Example Features |
|----------|-----------|------------------|
| Hotel (slow) | Daily | avg_review_score, cancellation_rate_30d, photo_quality_score |
| Hotel (fast) | 5-min | current_availability, demand_boost, price_competitiveness |
| User (slow) | Hourly | preferred_star_rating, avg_booking_price, destination_embedding |
| User (fast) | Real-time | session_hotel_views, current_search_context |
| Context | Per-request | days_to_check_in, is_holiday, search_time_of_day |

### Top 4 Feature Store Pitfalls

```
1. TRAINING/SERVING SKEW
   Computing features differently at training vs. serving time.
   #1 cause of production model degradation. Fix: same code, same store.

2. DATA LEAKAGE (no point-in-time join)
   Using review_count as of today to predict a booking from 6 months ago.
   The model 'knows the future' → inflated offline metrics, terrible online.

3. STALE FEATURES IN REAL-TIME MODELS
   Pricing model using 24-hour-old competitor prices → wrong decisions.
   Fix: monitor feature freshness per-feature with GoFresh-style SLA.

4. COLD START FOR NEW HOTELS
   New hotel has no historical features → model scores it poorly.
   Fix: city-level or category-level average features as priors.
```


In [ ]:
# ============================================================
# Point-in-Time Correctness — The Most Critical Feature Store Concept
# ============================================================
import pandas as pd
import numpy as np

# Simulate bookings (labels)
bookings_df = pd.DataFrame({
    'booking_id':        ['B001', 'B002', 'B003', 'B004'],
    'hotel_id':          [101,    101,    202,    202],
    'booking_timestamp': pd.to_datetime(['2025-01-15', '2025-03-01', '2025-02-10', '2025-04-05']),
    'booked':            [1, 1, 0, 1],  # label
})

# Simulate daily hotel feature snapshots (review count changes over time)
review_features = pd.DataFrame({
    'hotel_id':          [101,  101,  101,  202,  202,  202],
    'feature_timestamp': pd.to_datetime(['2025-01-01', '2025-02-01', '2025-04-01',
                                         '2025-01-01', '2025-02-15', '2025-04-01']),
    'review_count':      [50,   55,   70,   200,  210,  230],
})

print('=== WRONG: Using latest feature value (data leakage!) ===')
latest_features = review_features.groupby('hotel_id')['review_count'].max().reset_index()
wrong_join = bookings_df.merge(latest_features, on='hotel_id')
print(wrong_join[['booking_id', 'hotel_id', 'booking_timestamp', 'review_count', 'booked']])
print('  hotel 101 booking on 2025-01-15 sees review_count=70 (computed on 2025-04-01 — FUTURE!)')

print('\n=== CORRECT: Point-in-Time join ===')
def pit_join(labels_df, features_df, entity_col, label_ts_col, feature_ts_col):
    result_rows = []
    for _, label_row in labels_df.iterrows():
        # Get all feature snapshots for this entity BEFORE the label event
        valid = features_df[
            (features_df[entity_col] == label_row[entity_col]) &
            (features_df[feature_ts_col] <= label_row[label_ts_col])
        ]
        if not valid.empty:
            # Take the most recent one
            latest = valid.sort_values(feature_ts_col).iloc[-1]
            row = label_row.to_dict()
            row['review_count'] = latest['review_count']
            row['feature_as_of'] = latest[feature_ts_col]
            result_rows.append(row)
    return pd.DataFrame(result_rows)

correct = pit_join(bookings_df, review_features, 'hotel_id', 'booking_timestamp', 'feature_timestamp')
print(correct[['booking_id', 'hotel_id', 'booking_timestamp', 'feature_as_of', 'review_count', 'booked']])
print('\n  hotel 101 booking on 2025-01-15 correctly uses review_count=50 (as of 2025-01-01)')
print('  This is what the model could have ACTUALLY known at booking time.')


---
<a id='9'></a>
## 9. Data Warehouse & OLAP: StarRocks / Vertica / Impala

Agoda uses **StarRocks** as their primary OLAP engine.

### Why StarRocks?

| Feature | StarRocks | Vertica | Impala |
|---------|-----------|---------|--------|
| Query latency | Sub-second (vectorized MPP) | Seconds | Seconds |
| Real-time ingestion | ✅ Native Kafka → StarRocks | ❌ Batch only | ❌ Batch only |
| Concurrent queries | High | Medium | Medium |
| License | Open source | Commercial | Open source |
| Colocate join | ✅ No shuffle needed | ❌ | ❌ |

### StarRocks Table Types — Key Interview Topic

```sql
-- 1. DUPLICATE KEY — keeps all rows. Use for raw event logs.
CREATE TABLE search_events (
    event_id    BIGINT,
    hotel_id    BIGINT,
    user_id     BIGINT,
    search_ts   DATETIME
) DUPLICATE KEY(event_id)
  DISTRIBUTED BY HASH(hotel_id) BUCKETS 256;

-- 2. AGGREGATE KEY — pre-aggregates on insert. Use for metrics tables.
CREATE TABLE hotel_daily_metrics (
    hotel_id      BIGINT,
    metric_date   DATE,
    search_count  BIGINT SUM,       -- auto-summed on insert
    booking_count BIGINT SUM,
    revenue       DECIMAL(15,2) SUM,
    avg_price     DECIMAL(10,2) REPLACE  -- keeps latest value
) AGGREGATE KEY(hotel_id, metric_date)
  DISTRIBUTED BY HASH(hotel_id) BUCKETS 128;
-- New inserts auto-update aggregated values — no ETL merge needed!

-- 3. UNIQUE KEY — upsert semantics. Use for dimension tables.
CREATE TABLE hotels (
    hotel_id     BIGINT,
    name         VARCHAR(255),
    star_rating  TINYINT,
    country_code CHAR(2),
    updated_at   DATETIME
) UNIQUE KEY(hotel_id)
  DISTRIBUTED BY HASH(hotel_id) BUCKETS 128;
-- Updates overwrite existing rows — CDC-friendly
```

### Query Optimization Patterns

```sql
-- SLOW: Full scan, no partition pruning
SELECT country_code, SUM(total_price)
FROM bookings JOIN hotels USING (hotel_id)
WHERE MONTH(created_at) = 6;  -- function on partition key = FULL SCAN

-- FAST: Partition pruning + materialized view
SELECT country_code, SUM(total_price)
FROM bookings JOIN hotels USING (hotel_id)
WHERE created_at BETWEEN '2025-06-01' AND '2025-06-30';  -- direct comparison

-- EVEN FASTER: Materialized view pre-computes this pattern
CREATE MATERIALIZED VIEW mv_monthly_revenue
DISTRIBUTED BY HASH(booking_month, country_code)
AS
SELECT DATE_TRUNC('month', b.created_at) AS booking_month,
       h.country_code,
       SUM(b.total_price)  AS gross_revenue,
       COUNT(*)            AS booking_count
FROM bookings b JOIN hotels h USING (hotel_id)
WHERE b.status = 'CONFIRMED'
GROUP BY 1, 2;
-- Query planner auto-routes matching queries to this MV
```


In [ ]:
# ============================================================
# Indexing Strategy Decision Tree for OTA
# ============================================================

index_guide = {
    'Bitmap Index': {
        'use_for': 'Low-cardinality filter columns',
        'examples': ['star_rating (1-5)', 'room_type (5-10 values)', 'country_code (195 values)'],
        'how': 'Bitmap per distinct value. AND/OR operations are bit-level (blazing fast)',
        'avoid': 'High-cardinality columns (hotel_id, user_id) — too many bitmaps',
    },
    'Bloom Filter Index': {
        'use_for': 'High-cardinality, exact-match queries',
        'examples': ['hotel_id in large fact tables', 'booking_id lookups'],
        'how': 'Probabilistic structure: "does this value exist in block?" Skips irrelevant blocks.',
        'avoid': 'Range queries (> < BETWEEN) — bloom filters only help with =',
    },
    'Zone Map (Min/Max) Index': {
        'use_for': 'Range queries on sorted/clustered columns',
        'examples': ['check_in_date ranges', 'price ranges'],
        'how': 'Each column block stores min/max. WHERE price < 1000 skips blocks where min > 1000.',
        'avoid': 'Columns with no natural ordering or high randomness',
    },
    'Inverted Index (Elasticsearch)': {
        'use_for': 'Full-text search and geo queries',
        'examples': ['hotel name/description search', 'hotels near lat/lon', 'amenity text'],
        'how': 'Term → document list mapping. BKTree/geohash for geo proximity.',
        'avoid': 'Analytical aggregations — Elasticsearch is not an OLAP engine',
    },
    'Materialized View': {
        'use_for': 'Frequently-run expensive aggregations',
        'examples': ['Daily revenue by country', 'Weekly hotel conversion rates', 'Monthly demand forecasts'],
        'how': 'Pre-compute and store result. Async refresh in StarRocks.',
        'avoid': 'Highly dynamic queries with many filter combinations',
    },
}

for index_type, details in index_guide.items():
    print(f'=== {index_type} ===')
    for k, v in details.items():
        if isinstance(v, list):
            print(f'  {k}: {', '.join(v)}')
        else:
            print(f'  {k}: {v}')
    print()

print('=== Back-of-Envelope: Partition Pruning Impact ===')
total_rows = 13_000_000_000
partitions = 365
rows_per_partition = total_rows // partitions
print(f'Total price_rates rows: {total_rows:,}')
print(f'Partitions by check_in_date: {partitions}')
print(f'Rows per partition (1 day): {rows_per_partition:,}')
print(f'Query for one date scans: {rows_per_partition/total_rows*100:.2f}% of data')
print(f'Speedup vs full scan: {partitions}x')


---
<a id='10'></a>
## 10. Data Quality, Observability & SLAs

Agoda's FINUDP uses a **3-layer quality framework**: automated validation + ML anomaly detection + data contracts.

### The Three Layers

```
Layer 1 — Schema Validation (catches format/type errors)
  Null checks on required fields
  Type validation (price must be numeric, date must be valid)
  Range checks (price > 0, star_rating between 1 and 5)
  Referential integrity (hotel_id must exist in hotels table)

Layer 2 — Business Logic Validation (catches semantic errors)
  Revenue shouldn't drop > 40% day-over-day without event explanation
  Booking count per hotel shouldn't exceed room_count × nights
  Prices shouldn't exceed 10× the historical 30-day average
  Deduplication: same event_id should not appear twice

Layer 3 — ML Anomaly Detection (catches distributional shifts)
  Z-score / IQR on metric time-series
  Prophet for seasonality-aware anomaly detection
  Isolation Forest for multivariate anomalies

AGODA's FINUDP rule: auto-HALT pipeline on Layer 1 or Layer 2 critical failure.
Never process potentially incorrect financial data downstream.
```

### Data Freshness SLAs (GoFresh Pattern)

| Table | SLA | Severity on miss |
|-------|-----|------------------|
| hotel.prices (streaming) | 10 min | CRITICAL → page on-call |
| financial_unified (FINUDP) | 90 min | CRITICAL → page on-call |
| booking_events | 5 min | CRITICAL → page on-call |
| hotel_features_daily | 120 min | WARNING → Slack alert |
| ml_training_dataset | 24 hr | WARNING → Slack alert |

### Data Contracts

```yaml
# data_contract_hotel_prices.yaml
name: hotel.prices
version: '2.1.0'
owner: pricing-team
consumers: [ml-platform, bi-team, finance-team]

schema:
  - hotel_id:          {type: int64,   required: true}
  - price:             {type: double,  required: true, min: 0.01, max: 999999}
  - currency:          {type: string,  required: true, enum: [THB, USD, EUR, SGD]}
  - check_in_date:     {type: date,    required: true}
  - event_timestamp:   {type: int64,   required: true}

sla:
  freshness: 5 minutes
  completeness: 99.9%
  latency_p99: 10 seconds

breaking_change_policy: |
  30-day notice before removing/renaming fields.
  New required fields are breaking — add as optional first.
```


In [ ]:
# ============================================================
# Data Quality Framework — FINUDP Pattern
# ============================================================
from dataclasses import dataclass
from typing import List, Callable, Tuple
from enum import Enum
import pandas as pd, numpy as np, json

class Severity(Enum):
    WARNING  = 'WARNING'   # log + Slack
    ERROR    = 'ERROR'     # log + Slack + alert
    CRITICAL = 'CRITICAL'  # HALT pipeline (FINUDP approach)

@dataclass
class QualityCheck:
    name:     str
    severity: Severity
    check_fn: Callable  # (df) -> (passed: bool, message: str)


class DataQualityFramework:
    def __init__(self, pipeline_name: str):
        self.pipeline_name = pipeline_name
        self.checks: List[QualityCheck] = []

    def add(self, check: 'QualityCheck') -> 'DataQualityFramework':
        self.checks.append(check)
        return self

    def run(self, df: pd.DataFrame) -> bool:
        critical_failures = []
        for check in self.checks:
            passed, msg = check.check_fn(df)
            icon = '✅' if passed else {'CRITICAL': '🔴', 'ERROR': '🟠', 'WARNING': '🟡'}[check.severity.value]
            print(f'{icon} [{check.severity.value}] {check.name}: {msg}')
            if not passed:
                if check.severity == Severity.CRITICAL:
                    critical_failures.append(check.name)
                    self._alert(check.name, msg)
                elif check.severity == Severity.ERROR:
                    self._alert(check.name, msg)
        if critical_failures:
            print(f'\n🛑 Pipeline HALTED. Critical failures: {critical_failures}')
            return False
        print('\n✅ All checks passed. Pipeline proceeds.')
        return True

    def _alert(self, name, msg):
        print(f'   → [ALERT SENT] {self.pipeline_name} | {name}: {msg}')


# ── Check factories ──────────────────────────────────────────

def null_check(col: str, max_null_pct: float = 0.01):
    def check(df):
        pct = df[col].isna().mean() if col in df.columns else 0.0
        return pct <= max_null_pct, f'null_rate={pct:.3%} (threshold={max_null_pct:.3%})'
    return check

def range_check(col: str, lo: float, hi: float, max_bad_pct: float = 0.001):
    def check(df):
        if col not in df.columns: return True, 'column absent'
        bad = ((df[col] < lo) | (df[col] > hi)).mean()
        return bad <= max_bad_pct, f'{bad:.3%} rows outside [{lo}, {hi}]'
    return check

def revenue_drop_check(historical_avg: float, threshold: float = 0.40):
    def check(df):
        current = df['revenue'].sum() if 'revenue' in df.columns else 0
        drop = (historical_avg - current) / historical_avg if historical_avg else 0
        return drop <= threshold, f'current={current:,.0f} vs hist_avg={historical_avg:,.0f} drop={drop:.1%}'
    return check

def duplicate_check(col: str, max_dup_pct: float = 0.0):
    def check(df):
        dup_pct = df[col].duplicated().mean() if col in df.columns else 0
        return dup_pct <= max_dup_pct, f'duplicate_rate={dup_pct:.4%}'
    return check


# ── Run the framework ────────────────────────────────────────
np.random.seed(42)
sample = pd.DataFrame({
    'event_id':  range(10_000),
    'hotel_id':  np.random.randint(1, 50_000, 10_000),
    'price':     np.abs(np.random.normal(3000, 800, 10_000)),
    'revenue':   np.abs(np.random.normal(2_700_000, 200_000, 10_000)),
})

qf = DataQualityFramework('FINUDP-Daily')
qf.add(QualityCheck('hotel_id_nulls',   Severity.CRITICAL, null_check('hotel_id', 0.00)))
qf.add(QualityCheck('price_nulls',      Severity.CRITICAL, null_check('price', 0.01)))
qf.add(QualityCheck('price_range',      Severity.ERROR,    range_check('price', 0.01, 999_999)))
qf.add(QualityCheck('event_id_dedup',   Severity.CRITICAL, duplicate_check('event_id', 0.0)))
qf.add(QualityCheck('revenue_drop',     Severity.WARNING,  revenue_drop_check(27_000_000, 0.40)))

print('=== FINUDP Data Quality Run ===')
should_proceed = qf.run(sample)


---
<a id='11'></a>
## 11. Distributed Storage: Partitioning, Indexing, Caching

### Partitioning Strategies

| Strategy | How | When | OTA Example |
|----------|-----|------|-------------|
| Range | Split by value range | Time-series, date-based | bookings by created_at_month |
| Hash | `hash(key) % N` | Even distribution, point lookups | hotels by hash(hotel_id) |
| List | Explicit value list | Geographic/categorical | hotels by country_code |
| Composite | Range + Hash | Most real-world cases | prices by (country, RANGE date) |

### Multi-Layer Cache Design for Hotel Price Lookup

```
L1: In-process LRU cache     < 1 ms   top 10K hotels by traffic (covers 80% of requests)
L2: Redis Cluster            < 5 ms   hot window — 90-day prices (~194 GB)
L3: StarRocks                < 100ms  full date range, OLAP queries
L4: HDFS                     < 1 s    complete historical data

Cache-aside pattern (read path):
  1. Check L1 → hit: return immediately
  2. Check L2 (Redis) → hit: populate L1, return
  3. Query L3 (StarRocks) → populate L2+L1, return
  4. Miss all → return error / show unavailable

Invalidation:
  On price update event (from Kafka): delete L1 entry immediately
  Redis TTL = 5 min (acceptable staleness for search results)
  CRITICAL: availability (room count) must be invalidated on each booking
            to prevent double-booking. Use event-driven invalidation, not TTL.
```


In [ ]:
# ============================================================
# Consistent Hashing — Used in Redis Cluster & Distributed Caches
# ============================================================
import hashlib, bisect

class ConsistentHashRing:
    '''
    Maps keys to nodes using a virtual-node ring.
    When a node is added/removed, only ~1/N keys are remapped
    (vs naive modulo hashing which remaps nearly all keys).
    '''
    def __init__(self, replicas: int = 150):
        self.replicas = replicas  # virtual nodes per physical node
        self.ring = {}            # ring_position → node
        self.sorted_keys = []

    def add_node(self, node: str):
        for i in range(self.replicas):
            pos = self._hash(f'{node}:{i}')
            self.ring[pos] = node
            bisect.insort(self.sorted_keys, pos)

    def remove_node(self, node: str):
        for i in range(self.replicas):
            pos = self._hash(f'{node}:{i}')
            self.ring.pop(pos, None)
            idx = bisect.bisect_left(self.sorted_keys, pos)
            if idx < len(self.sorted_keys) and self.sorted_keys[idx] == pos:
                self.sorted_keys.pop(idx)

    def get_node(self, key: str) -> str:
        if not self.ring:
            raise RuntimeError('No nodes in ring')
        pos = self._hash(key)
        idx = bisect.bisect(self.sorted_keys, pos) % len(self.sorted_keys)
        return self.ring[self.sorted_keys[idx]]

    def _hash(self, key: str) -> int:
        return int(hashlib.md5(key.encode()).hexdigest(), 16)


# Demo: Price cache routing
ring = ConsistentHashRing(replicas=150)
for node in ['redis-1', 'redis-2', 'redis-3']:
    ring.add_node(node)

print('=== 3-node ring ===')
test_hotels = [12345, 99999, 50001, 777, 88888]
before = {}
for hid in test_hotels:
    node = ring.get_node(f'hotel:{hid}:prices')
    before[hid] = node
    print(f'  hotel {hid:>6} → {node}')

ring.add_node('redis-4')
print('\n=== After adding redis-4 (only ~25% of keys remapped) ===')
remapped = 0
for hid in test_hotels:
    node = ring.get_node(f'hotel:{hid}:prices')
    changed = '← REMAPPED' if node != before[hid] else ''
    if changed: remapped += 1
    print(f'  hotel {hid:>6} → {node} {changed}')
print(f'\nRemapped: {remapped}/{len(test_hotels)} = {remapped/len(test_hotels):.0%}')
print('(Naive modulo hashing would remap ~75% of keys)')


---
<a id='12'></a>
## 12. Operational Excellence: Monitoring, DR, Graceful Degradation

### The Four Golden Signals (for every data pipeline)

```
1. LATENCY      How long does processing take?  (p50, p95, p99)
2. THROUGHPUT   How many events/sec processed?  (should match input rate)
3. ERROR RATE   What % of events fail?          (alert if > 0.1%)
4. SATURATION   How full are queues/buffers?    (Kafka lag, memory usage)
```

### Agoda's Monitoring Stack

```
Application logs  → Kafka → Elasticsearch → Kibana  (log search)
Kafka metrics     → JMXTrans → Graphite → Grafana    (broker health)
Custom SLA        → White Falcon (in-house TSDB)     (freshness)
Data freshness    → GoFresh → Slack / PagerDuty      (SLA alerts)
ML model health   → custom Kafka pipeline → HDFS     (drift detection)
```

### Disaster Recovery Matrix

| System | RTO | RPO | Strategy |
|--------|-----|-----|----------|
| Booking system | < 5 min | **Zero** | Active-Active multi-DC + Kafka MirrorMaker |
| Price updates | < 15 min | < 5 min | Active-Passive + replay Kafka from checkpoint |
| Analytics pipeline | < 2 hr | < 1 hr | Resume Spark from HDFS checkpoint |
| ML feature store | < 4 hr | < 24 hr | Recompute from HDFS snapshots |
| Historical data lake | < 24 hr | < 24 hr | HDFS replication + cold backup |

### Active-Active vs Active-Passive

```
ACTIVE-ACTIVE (booking system):
  DC Bangkok ◄──────────────────────► DC Singapore
              Kafka MirrorMaker 2
              (all topics replicated in real-time)
  Both DCs accept writes. Conflict resolution: booking_id UUID dedup.
  Failover: DNS switch. No data loss.

ACTIVE-PASSIVE (analytics lake):
  DC Bangkok (Primary) → HDFS replication → DC Singapore (Standby)
  Failover: promote standby, restart Spark jobs from last checkpoint.
  RPO = replication lag (typically < 1 min).
```


In [ ]:
# ============================================================
# Graceful Degradation + Circuit Breaker — Production Patterns
# ============================================================
import time
from enum import Enum

class CircuitState(Enum):
    CLOSED    = 'CLOSED'     # Normal — all requests pass through
    OPEN      = 'OPEN'       # Failing — fast-fail all requests
    HALF_OPEN = 'HALF_OPEN'  # Testing recovery

class CircuitBreaker:
    '''
    Prevents cascading failures.
    If ML ranking service keeps timing out, stop calling it.
    Serve rule-based results instead. Try again after recovery_timeout.
    '''
    def __init__(self, name: str, failure_threshold: int = 5, recovery_timeout: float = 60):
        self.name = name
        self.failure_threshold = failure_threshold
        self.recovery_timeout  = recovery_timeout
        self.failure_count     = 0
        self.last_failure_time = None
        self.state             = CircuitState.CLOSED

    def call(self, fn, *args, **kwargs):
        if self.state == CircuitState.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = CircuitState.HALF_OPEN
                print(f'[{self.name}] Circuit HALF_OPEN — testing recovery')
            else:
                raise RuntimeError(f'[{self.name}] Circuit OPEN — fast fail')
        try:
            result = fn(*args, **kwargs)
            if self.state == CircuitState.HALF_OPEN:
                self._reset()
            return result
        except Exception as e:
            self._on_failure()
            raise

    def _on_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
            print(f'[{self.name}] Circuit OPEN after {self.failure_count} failures')

    def _reset(self):
        self.failure_count = 0
        self.state = CircuitState.CLOSED
        print(f'[{self.name}] Circuit CLOSED — service recovered')


# Graceful degradation for hotel search
class HotelSearchService:
    def __init__(self):
        self.ml_cb    = CircuitBreaker('ml-ranking', failure_threshold=3)
        self.redis_cb = CircuitBreaker('redis-price', failure_threshold=3)

    def search(self, query: dict) -> list:
        hotels = self._elasticsearch_search(query)  # always runs

        # Try: ML ranking with live prices
        try:
            prices = self.redis_cb.call(self._redis_prices, hotels)
            ranked = self.ml_cb.call(self._ml_rank, hotels, prices, query)
            return ranked
        except RuntimeError as e:
            if 'ml-ranking' in str(e):
                print('  Degraded mode: rule-based ranking (no personalization)')
                prices = self._starrocks_prices(hotels)
                return self._rule_based_rank(hotels, prices)
            elif 'redis-price' in str(e):
                print('  Degraded mode: using StarRocks prices (may be 5min stale)')
                prices = self._starrocks_prices(hotels)
                return self._ml_rank(hotels, prices, query)

    # Stub implementations
    def _elasticsearch_search(self, q): return [{'hotel_id': i, 'score': 0.9-i*0.01} for i in range(5)]
    def _redis_prices(self, hotels): return {h['hotel_id']: 3000+h['hotel_id']*10 for h in hotels}
    def _starrocks_prices(self, hotels): return {h['hotel_id']: 3100+h['hotel_id']*10 for h in hotels}
    def _ml_rank(self, hotels, prices, query):
        return sorted(hotels, key=lambda h: -h['score'])
    def _rule_based_rank(self, hotels, prices):
        return sorted(hotels, key=lambda h: prices.get(h['hotel_id'], 999_999))


svc = HotelSearchService()
print('=== Normal operation ===')
results = svc.search({'destination': 'Bangkok', 'check_in': '2025-06-01'})
print(f'Results: {[r["hotel_id"] for r in results]}')

# Simulate ML service going down
print('\n=== Simulating ML ranking failure ===')
def failing_ml(*args, **kwargs): raise Exception('ML service timeout')
for i in range(4):
    try:
        svc.ml_cb.call(failing_ml)
    except:
        pass

print('\n=== Search with ML circuit open (fallback to rule-based) ===')
results = svc.search({'destination': 'Bangkok', 'check_in': '2025-06-01'})
print(f'Results: {[r["hotel_id"] for r in results]}')


---
<a id='13'></a>
## 13. Common Mistakes, Tricks & Edge Cases at OTA Scale

### 🔴 Top 10 Interview Mistakes

| # | Mistake | Better Approach |
|---|---------|----------------|
| 1 | Designing for 100 QPS when the problem implies 1M+ | Always ask scale upfront; size every component explicitly |
| 2 | Ignoring data skew | Address hot hotels (top 1% get 50% traffic) — salting, AQE, broadcast |
| 3 | Single Kafka cluster for everything | Cluster separation by use-case — blast radius isolation |
| 4 | Strong consistency everywhere | Match consistency to business need; prices tolerate eventual |
| 5 | No checkpoint mention in streaming | Always state checkpoint location — it's the fault-tolerance mechanism |
| 6 | Forgetting exactly-once for financial events | Booking/payment = Kafka transactions + idempotency key |
| 7 | No data quality layer | Add validation gates; describe auto-halt on critical failure |
| 8 | No monitoring/alerting | Four golden signals + GoFresh freshness SLA monitoring |
| 9 | Over-engineering from the start | Start simple → identify bottleneck → add complexity only where needed |
| 10 | Ignoring schema evolution | Avro + Schema Registry + backward compatibility |

### 🧩 OTA-Specific Edge Cases

```
EDGE CASE 1: Hotel shows available in search but is fully booked at checkout
  Root cause: Availability cache stale between search and booking confirmation
  Fix: Quote ID — lock price+availability for 15 min when checkout starts
       Final availability re-check against Cassandra before booking confirmation

EDGE CASE 2: Duplicate booking (user double-clicked or network timeout)
  Root cause: Request timeout causes retry → two bookings for same session
  Fix: Idempotency key (client-generated UUID) sent with every booking request
       Server stores processed keys in Redis with 24h TTL
       Second request with same key returns first booking silently

EDGE CASE 3: Price shown differs from price at checkout
  Root cause: Price updated between search cache and checkout page
  Fix: Quote ID stores locked price in Redis for 15 min
       Checkout uses quoted price, not live price

EDGE CASE 4: Currency conversion race condition
  Root cause: Exchange rate changes between search (USD display) and booking (THB charge)
  Fix: Always store and transact in supplier currency (THB)
       Apply display-currency conversion at render time only
       Price lock includes exchange rate at time of quote

EDGE CASE 5: Spark job wrong results due to timezone
  Agoda HQ is Bangkok (UTC+7). Booking at 23:00 BKK = 16:00 UTC
  Fix: ALWAYS store timestamps in UTC. Convert at display only.
       spark.conf.set('spark.sql.session.timeZone', 'UTC')

EDGE CASE 6: Backfill corrupts real-time data
  Backfill writes to same table/partition as live pipeline → overwrites fresh data
  Fix: Backfill writes to separate partition suffix (_backfill)
       Merge explicitly after validation; delete backfill partition

EDGE CASE 7: Holiday season traffic spike (10x normal)
  Kafka producer rate exceeds consumer capacity → lag spikes
  Fix: Pre-scale consumer groups 48h before known events (Songkran, Chinese NY)
       Auto-scaling based on Kafka lag metric in Kubernetes HPA
```


In [ ]:
# ============================================================
# Back-of-Envelope Calculations — Do These Live in the Interview
# ============================================================

print('=== OTA Scale Estimates ===')
print()

# Hotels & Price Points
hotels = 3_600_000
dates  = 365
room_types = 10
bytes_per_record = 200

total_price_pts = hotels * dates * room_types
total_size_gb   = total_price_pts * bytes_per_record / 1e9
hot_window_keys = int(hotels * (90/365) * 0.30 * room_types)
hot_window_gb   = hot_window_keys * bytes_per_record / 1e9

print(f'Hotels:                   {hotels:>15,}')
print(f'Total price combinations: {total_price_pts:>15,}')
print(f'Raw storage:              {total_size_gb:>14.0f} GB  ← too big for Redis')
print(f'Hot window (90d, 30% avail): {hot_window_gb:>10.0f} GB  ← feasible in Redis')
print()

# Kafka throughput
kafka_events_day = 1_800_000_000_000  # 1.8T
kafka_eps        = kafka_events_day / 86_400
kafka_mps        = kafka_eps / 1_000_000
print(f'Kafka events/day:         {kafka_events_day:>15,}')
print(f'Kafka events/sec:         {kafka_eps:>15,.0f}')
print(f'Kafka events/sec (M):     {kafka_mps:>14.1f} M/s')
print()

# Search QPS
searches_day = 50_000_000  # 50M searches/day estimate
search_qps_avg  = searches_day / 86_400
search_qps_peak = search_qps_avg * 10  # 10x peak factor
print(f'Searches/day:             {searches_day:>15,}')
print(f'Search QPS (avg):         {search_qps_avg:>15.0f}')
print(f'Search QPS (10x peak):    {search_qps_peak:>15.0f}')
print()

# Benchmark reference numbers
print('=== Component Throughput Benchmarks (memorize these) ===')
benchmarks = [
    ('Redis single node',          '100 K ops/sec'),
    ('Redis Cluster (10 nodes)',   '1 M ops/sec'),
    ('Kafka partition throughput', '1–10 MB/sec'),
    ('Kafka cluster write (100p)', '~1 GB/sec'),
    ('Spark task overhead',        '~100 ms/task'),
    ('SSD random read',            '100 K IOPS = 0.1 ms'),
    ('HDD random read',            '100 IOPS = 10 ms'),
    ('Same-DC network RTT',        '< 1 ms'),
    ('Cross-DC network RTT',       '50–100 ms'),
    ('Elasticsearch search p99',   '20–50 ms'),
    ('StarRocks analytical query', '100 ms – 2 s'),
]
for component, value in benchmarks:
    print(f'  {component:<45} → {value}')


---
<a id='14'></a>
## 14. System Design Templates — Whiteboard Approach

### Universal 60-Minute Framework

```
⏱️  0– 5 min  CLARIFY
             Ask the 8 clarifying questions from Section 2.
             Agree on scope. Define success metrics.

⏱️  5–10 min  ESTIMATE
             Back-of-envelope: QPS, storage, bandwidth.
             Identify which tier (L1/L2/L3) each number implies.

⏱️ 10–20 min  HIGH-LEVEL ARCHITECTURE
             Draw the boxes: source → ingest → process → store → serve.
             Name each component (Kafka/Spark/Redis/StarRocks).
             Mention the SLA for each hop.

⏱️ 20–40 min  DEEP DIVE (pick 2–3 hard components)
             Partitioning strategy for Kafka/storage.
             Exactly-once semantics if financial data.
             Cache invalidation strategy.
             Data quality gates.

⏱️ 40–50 min  TRADE-OFFS & FAILURE MODES
             For each major decision: 'I chose X over Y because...'
             Walk through 2–3 failure scenarios and mitigations.
             Mention graceful degradation.

⏱️ 50–60 min  OPERATIONS & EVOLUTION
             Monitoring: 4 golden signals, GoFresh freshness.
             Scaling: what breaks first at 10× load?
             How do we evolve the schema without downtime?
```

### Template: Real-Time Data Pipeline (fits 80% of questions)

```
[Trigger: User action / Supplier event / Time-based]
    │
    ▼ (Kafka — partition by entity_id for ordering)
[Ingestion: Kafka topic]
    │
    ▼ (Spark Structured Streaming — checkpoint to HDFS)
[Stream Processing: windowed aggregation + enrichment + quality check]
    │
    ├──► [Hot serving: Redis — TTL = acceptable staleness]
    └──► [Analytics: StarRocks — materialized views for dashboards]
                │
                └──► [ML Feature Store: HDFS Parquet (offline) + Redis (online)]
    │
    ▼
[Monitoring: Grafana dashboard]
    ├── Consumer lag (saturation)
    ├── Processing throughput (throughput)
    ├── Error rate (error rate)
    ├── End-to-end latency (latency)
    └── Table freshness (GoFresh SLA)
```

### How to Handle 'Improve This System' Questions (Agoda's Platform Round)

```
STEP 1: Understand current state
  'What's the current throughput and p99 latency?'
  'What's the biggest pain point — reliability, freshness, or cost?'

STEP 2: Identify bottleneck type
  CPU-bound   → parallelize, vectorize, optimize algorithm
  Memory-bound → tune heap, use off-heap, spill to disk
  I/O-bound   → batching, async I/O, caching, better indexes
  Network-bound → compression, reduce data movement, co-location
  Lock contention → reduce transaction scope, optimistic locking

STEP 3: Propose graduated improvements
  Quick wins:    add covering index, enable AQE, increase Redis TTL
  Medium effort: repartition strategy, consumer scaling, broadcast joins
  Big effort:    architectural change (e.g., add streaming layer, adopt StarRocks)

STEP 4: Measure success
  Define before/after metrics for EACH improvement
  A/B test changes in production (Agoda runs 1000 experiments!)
```


---
<a id='15'></a>
## 15. Mock Interview Q&A Bank

---

### 🔥 Q1: Design a Dynamic Pricing Data Pipeline for Agoda

**Clarify first:**
- 3.6M hotels, 365 dates, 10 room types = 13B price points
- Price updates every 5 min from supplier APIs
- Serve prices in < 5ms for search page

**Architecture:**
1. Supplier feeds → Kafka `hotel.prices` (partitioned by hotel_id)
2. Spark Streaming: 5-min micro-batch → compute min price per (hotel, date, room_type)
3. Write hot window (90d × 30% avail = ~194GB) to Redis HASH per hotel+date
4. Write full history to HDFS Parquet → StarRocks for analytics
5. Anomaly detection: price > 5× 30-day avg → quarantine + alert

**Trade-offs stated:**
- Eventual consistency (5-min stale) acceptable for search; exact price locked at checkout (Quote ID)
- Micro-batch over true streaming: suppliers don't update faster; reduces write amplification

---

### 🔥 Q2: How would you reduce the FINUDP pipeline from 5 hours to 30 minutes?

*This shows you read their engineering blog.*

**Answer (ordered by impact):**
1. **Parallel sub-pipelines** — run Finance, Planning, Ledger ETLs in parallel instead of serial
2. **Broadcast joins** — hotel_meta (~720MB) broadcast to every executor; eliminates shuffle join
3. **Partition pruning** — ensure WHERE clauses use raw partition column (no functions)
4. **AQE** — enable Spark Adaptive Query Execution for automatic skew handling
5. **Cache reused DataFrames** — `.cache()` tables joined multiple times
6. **Columnar reads** — read Parquet instead of CSV; predicate pushdown reduces I/O
7. **Right-size shuffle partitions** — profile with Spark UI, increase from default 200

---

### 🔥 Q3: What happens when your streaming pipeline falls 1 hour behind?

**This tests operational thinking — interviewers love this.**

```
1. DETECT:
   Grafana alert: consumer lag > 1M events for > 5 min → PagerDuty page

2. DIAGNOSE (in this order):
   a. Slow processing? Spark UI → which stage is slowest?
   b. Kafka broker issue? Check broker metrics (disk I/O, network saturation)
   c. Downstream bottleneck? Redis / HDFS write throughput
   d. Data spike? Look at incoming event rate vs. recent baseline

3. MITIGATE IMMEDIATELY:
   Scale up Spark executor count (dynamic allocation or manual)
   Reduce processing complexity temporarily — skip enrichment, just catch up
   If Redis is bottleneck: switch to async write (buffer results, write in batch)

4. AFTER CATCH-UP:
   Root cause analysis
   Adjust maxOffsetsPerTrigger if data spike was cause
   Increase partition count if sustained higher throughput is expected

5. PREVENT:
   Monitor lag trend (not just absolute value) — alert on rate of increase
   Pre-scale before Agoda's known high-traffic events (Songkran, Chinese New Year)
```

---

### 🔥 Q4: Design an A/B Testing Platform (Agoda runs 1000 simultaneous experiments)

**Key components:**

```
1. Assignment Service
   hash(user_id + experiment_id) → deterministic treatment/control assignment
   Store in Redis: exp:{id}:user:{user_id} → 'treatment'  (TTL = experiment duration)
   Same user always gets same bucket (hash consistency)

2. Event Collection
   Every user action tagged with active experiments list
   Kafka event: {user_id, action, value, experiments: ['exp_1:T', 'exp_2:C']}

3. Metrics Pipeline (Spark batch, daily)
   JOIN(events, assignments) → aggregate per (experiment × treatment)
   Compute: conversion_rate, revenue_per_user, booking_count

4. Statistical Analysis
   t-test / Mann-Whitney U for significance
   Bonferroni correction for 1000 simultaneous experiments (multiple testing!)
   CUPED variance reduction for faster decisions

5. Results Dashboard
   StarRocks → Metabase: live experiment metrics
   Auto-flag statistically significant results for product review
```

---

### 🔥 Q5: How do you handle exactly-once booking at Agoda's scale?

**Multi-layer approach:**

```
Layer 1: Idempotency Key (client side)
  Client generates UUID booking_request_id before sending
  On timeout/retry, client resends with SAME UUID
  Server checks Redis: IF seen_requests:{uuid} EXISTS → return original result

Layer 2: Kafka Transactions (producer side)
  enable.idempotence = true
  transactional.id = booking-producer-{instance}
  Exactly-once: Kafka guarantees no duplicate messages even on broker failure

Layer 3: Database Idempotency (consumer side)
  INSERT INTO bookings ... ON CONFLICT (booking_request_id) DO NOTHING
  Or: check before insert within a transaction

Layer 4: Saga Pattern for distributed booking
  Booking = (reserve inventory → charge payment → confirm booking)
  Each step is reversible (compensating transaction)
  If payment fails: release inventory reservation
```

---

### 📝 Behavioral Supplement

```
'Tell me about a time you redesigned a data pipeline at scale.'
→ Use Tiket.com's Orion pricing platform (RL + gradient boosting) or LLM chatbot.
  Structure: Situation → Task → Action → Result (STAR).
  Quantify impact: 'reduced pricing latency by X%', 'handled Y% more traffic'.

'How do you handle conflicting priorities between data freshness and cost?'
→ Define SLA per consumer (ML model vs. BI dashboard vs. finance report).
  Optimize cost within each SLA. Price freshness SLA = 5 min.
  Review score SLA = 24 hr. Different pipelines, different costs.

'Describe a data quality issue you caught before production.'
→ Describe: what you monitored, how you detected it, immediate mitigation, long-term fix.
  Ideal answer: ML anomaly detection caught 40% revenue drop due to timezone bug.
```

---

## 🎯 Final Checklist Before Your Interview

- [ ] Know Agoda's real tech stack cold: Kafka · Spark · StarRocks · MLflow · Kubeflow · HDFS
- [ ] Can explain the 2-Step Logging Architecture and why it outperforms direct Kafka writes
- [ ] Know FINUDP: unified pipeline, 3-layer quality, hourly SLA, auto-halt on failure
- [ ] Can estimate 13B price points, 1.8T events/day, 194GB hot Redis window on the spot
- [ ] Can design any OTA system at Agoda's scale within 60 minutes
- [ ] Know Lambda vs. Kappa, exactly-once semantics, consistent hashing, PIT joins
- [ ] Prepared 3 concrete Tiket.com examples: Hotel Ranking, Orion Pricing, LLM Chatbot
- [ ] Can describe graceful degradation for every component you design
- [ ] Can discuss monitoring, SLAs, and DR for any system
- [ ] Ask clarifying questions BEFORE diving into design — always
